# 🔀 Silver - Expansão de Arrays em Tabelas Derivadas

## 🎯 Objetivo
Transformar a tabela **`sales_orders`** (com arrays nested) em **tabelas normalizadas** para facilitar análises granulares.

## 📊 Problema: Arrays de Tamanho Variável

A tabela `sales_orders` tem arrays com **cardinalidade N**:
- Cada pedido tem um **número DIFERENTE de produtos** (1 a 4 itens)
- Cada pedido tem um **número DIFERENTE de clicks** (1 a 10 clicks)

### ❌ **Por que NÃO expandir em colunas?**

```
sales_orders:
ORDER_NUMBER | PRODUCT_1_ID | PRODUCT_1_NAME | PRODUCT_2_ID | PRODUCT_2_NAME | PRODUCT_3_ID | ...
317568014    | AVpfuJ4pi... | Rony Mini-Sys  | AVpe6jFBi... | Aeon Screen    | AVpfIODe1... | ...
317568015    | AVpfdBS41... | Mogitech Wheel | NULL         | NULL           | NULL         | ...
```

**Problemas**:
- 🚫 MUITAS colunas (4 produtos × 7 campos = 28 colunas extras!)
- 🚫 MUITOS NULLs (pedido com 1 item tem 21 colunas vazias = 75% desperdício)
- 🚫 Limite de colunas (e se aparecer pedido com 10 produtos?)
- 🚫 Análises IMPOSSÍVEIS (`SUM(price * qty)` por produto?)

### ✅ **Solução: Explodir em Tabelas Separadas (Normalização)**

```
sales_order_items (1 linha = 1 produto):
ORDER_NUMBER | PRODUCT_ID    | PRODUCT_NAME      | PRODUCT_PRICE | QUANTITY | ITEM_TOTAL
317568014    | AVpfuJ4pi...  | Rony Mini-System  | 993           | 3        | 2979
317568014    | AVpe6jFBi...  | Aeon Screen       | 218           | 3        | 654
317568014    | AVpfIODe1...  | Cyber-shot Camera | 448           | 2        | 896
317568015    | AVpfdBS41...  | Mogitech Wheel    | 293           | 4        | 1172
```

**Vantagens**:
- ✅ Sem NULLs
- ✅ Análises simples: `SELECT SUM(ITEM_TOTAL) GROUP BY PRODUCT_ID`
- ✅ JOINs diretos: `JOIN products ON items.PRODUCT_ID = products.PRODUCT_ID`
- ✅ Escalável (1 ou 100 produtos, não importa)

## 🏗️ Arquitetura Criada

```
┌─────────────────────────────────────┐
│   sales_orders (FATO AGREGADO)      │  ← 1 linha = 1 pedido
│   - ORDER_NUMBER (PK)               │     4.000 registros
│   - CUSTOMER_ID                     │
│   - ORDER_DATETIME                  │
│   - ORDERED_PRODUCTS (array)        │  ← Array mantido
└──────────────┬──────────────────────┘
               │
               │ FK: ORDER_NUMBER
               │
       ┌───────┴────────┐
       │                │
       ▼                ▼
┌──────────────┐  ┌──────────────────┐
│ sales_order_ │  │ sales_order_     │
│ items        │  │ clicks           │
│              │  │                  │
│ 1 linha =    │  │ 1 linha =        │
│ 1 PRODUTO    │  │ 1 CLICK          │
└──────────────┘  └──────────────────┘
```

## 📋 Tabelas que Serão Criadas

| Tabela | Granularidade | Registros Estimados | FK |
|--------|---------------|---------------------|----|
| **sales_order_items** ⭐ | 1 linha = 1 item | ~8.000 | ORDER_NUMBER, PRODUCT_ID |
| **sales_order_clicks** | 1 linha = 1 click | ~16.000 | ORDER_NUMBER, CLICKED_PRODUCT_ID |

In [0]:
from pyspark.sql import functions as F, Window
import uuid
from datetime import datetime

# Configurações
CATALOG = "retail_dev"
SILVER_SCHEMA = "silver"
SOURCE_TABLE = "sales_orders"  # Tabela origem (com arrays)

# Tabelas derivadas que serão criadas
ITEMS_TABLE = "sales_order_items"
CLICKS_TABLE = "sales_order_clicks"

# ID único para este pipeline run
pipeline_run_id = str(uuid.uuid4())
processing_timestamp = datetime.now()

print(f"🔧 Configuração carregada:")
print(f"   📦 Tabela origem: {CATALOG}.{SILVER_SCHEMA}.{SOURCE_TABLE}")
print(f"   📤 Tabela destino 1: {CATALOG}.{SILVER_SCHEMA}.{ITEMS_TABLE}")
print(f"   📤 Tabela destino 2: {CATALOG}.{SILVER_SCHEMA}.{CLICKS_TABLE}")
print(f"   🆔 Pipeline Run ID: {pipeline_run_id}")
print(f"   ⏰ Timestamp: {processing_timestamp}")

## 📦 1️⃣ Explodir ORDERED_PRODUCTS → Tabela `sales_order_items` ⭐

### 🎯 Objetivo
Criar tabela com **uma linha por produto** em cada pedido.

### 📋 Transformações
1. **Explodir array**: `ORDERED_PRODUCTS` (array de structs)
2. **Flatten struct**: Extrair campos do produto (id, name, price, qty, unit, curr)
3. **Flatten nested**: Extrair `promotion_info` (struct dentro do struct)
4. **Colunas calculadas**:
   - `ITEM_TOTAL` = price × qty
   - `HAS_ITEM_PROMO` = Se tem promoção neste item
   - `ITEM_POSITION` = Posição do item no pedido (1, 2, 3...)

### 🔍 Estrutura dos Dados

**ANTES (array nested):**
```python
ORDERED_PRODUCTS: ARRAY<STRUCT<
    curr: STRING,
    id: STRING,
    name: STRING,
    price: LONG,
    qty: LONG,
    unit: STRING,
    promotion_info: STRUCT<  ← STRUCT NESTED!
        promo_disc: DOUBLE,
        promo_id: LONG,
        promo_item: STRING,
        promo_qty: LONG
    >
>>
```

**DEPOIS (tabela normalizada):**
```
COLUNAS: ORDER_NUMBER, CUSTOMER_ID, PRODUCT_ID, PRODUCT_NAME, 
         PRODUCT_PRICE, QUANTITY, UNIT, CURRENCY,
         PROMO_DISCOUNT, PROMO_ID, PROMO_ITEM, PROMO_QTY,
         ITEM_TOTAL, HAS_ITEM_PROMO, ITEM_POSITION
```

In [0]:
print("📦 1️⃣ Explodindo ORDERED_PRODUCTS → sales_order_items...")
print("=" * 80)

# Ler tabela Silver original
source_table_name = f"{CATALOG}.{SILVER_SCHEMA}.{SOURCE_TABLE}"
orders_df = spark.table(source_table_name)

print(f"\n📖 Lendo tabela: {source_table_name}")
initial_orders_count = orders_df.count()
print(f"   Total de pedidos: {initial_orders_count:,}")

# Explodir array ORDERED_PRODUCTS
# F.explode() transforma cada elemento do array em uma linha
items_df = orders_df.select(
    # --- COLUNAS DO PEDIDO (repetidas para cada item) ---
    "ORDER_NUMBER",
    "CUSTOMER_ID",
    "CUSTOMER_NAME",
    "ORDER_DATETIME",
    "NUMBER_OF_LINE_ITEMS",
    
    # --- EXPLODIR ARRAY → Cada produto vira uma linha ---
    F.explode("ORDERED_PRODUCTS").alias("product"),
    
    # --- AUDITORIA DA TABELA ORIGEM ---
    "DATA_QUALITY_SCORE",
    "PROCESSED_AT",
    "SOURCE_TABLE",
    "PIPELINE_RUN_ID"
).select(
    # Colunas do pedido
    "ORDER_NUMBER",
    "CUSTOMER_ID",
    "CUSTOMER_NAME",
    "ORDER_DATETIME",
    "NUMBER_OF_LINE_ITEMS",
    
    # --- FLATTEN DO STRUCT PRODUCT ---
    F.col("product.curr").alias("CURRENCY"),
    F.col("product.id").alias("PRODUCT_ID"),
    F.col("product.name").alias("PRODUCT_NAME"),
    F.col("product.price").alias("PRODUCT_PRICE"),
    F.col("product.qty").alias("QUANTITY"),
    F.col("product.unit").alias("UNIT"),
    
    # --- FLATTEN DO STRUCT NESTED PROMOTION_INFO ---
    F.col("product.promotion_info.promo_disc").alias("PROMO_DISCOUNT"),
    F.col("product.promotion_info.promo_id").alias("PROMO_ID"),
    F.col("product.promotion_info.promo_item").alias("PROMO_ITEM"),
    F.col("product.promotion_info.promo_qty").alias("PROMO_QTY"),
    
    # Auditoria
    "DATA_QUALITY_SCORE",
    "PROCESSED_AT",
    "SOURCE_TABLE",
    "PIPELINE_RUN_ID"
)

# Adicionar colunas calculadas
items_df = items_df.withColumn(
    # ITEM_TOTAL: price * qty (total deste item no pedido)
    "ITEM_TOTAL",
    F.coalesce(F.col("PRODUCT_PRICE"), F.lit(0)) * F.coalesce(F.col("QUANTITY"), F.lit(0))
).withColumn(
    # HAS_ITEM_PROMO: Se este item específico tem promoção
    "HAS_ITEM_PROMO",
    F.when(F.col("PROMO_ID").isNotNull(), True).otherwise(False)
).withColumn(
    # ITEM_POSITION: Posição do item no pedido (1, 2, 3...)
    "ITEM_POSITION",
    F.row_number().over(Window.partitionBy("ORDER_NUMBER").orderBy(F.monotonically_increasing_id()))
)

total_items = items_df.count()
total_orders = items_df.select("ORDER_NUMBER").distinct().count()

print(f"\n✅ ORDERED_PRODUCTS explodido com sucesso!")
print(f"\n📊 Estatísticas:")
print(f"   Total de itens (linhas): {total_items:,}")
print(f"   Total de pedidos únicos: {total_orders:,}")
print(f"   Média de itens por pedido: {total_items/total_orders:.2f}")

print(f"\n📋 Schema da tabela sales_order_items:")
items_df.printSchema()

print(f"\n📊 Amostra (10 primeiros itens):")
display(items_df.limit(10))

print(f"\n📊 Análise de promoções nos itens:")
with_promo = items_df.filter(F.col("HAS_ITEM_PROMO") == True).count()
without_promo = items_df.filter(F.col("HAS_ITEM_PROMO") == False).count()
print(f"   🎁 Itens COM promoção: {with_promo:,} ({(with_promo/total_items)*100:.2f}%)")
print(f"   📦 Itens SEM promoção: {without_promo:,} ({(without_promo/total_items)*100:.2f}%)")

print(f"\n📊 Top 5 produtos mais vendidos (por quantidade):")
display(
    items_df.groupBy("PRODUCT_ID", "PRODUCT_NAME")
    .agg(
        F.sum("QUANTITY").alias("total_qty_sold"),
        F.count("*").alias("times_ordered"),
        F.sum("ITEM_TOTAL").alias("total_revenue")
    )
    .orderBy(F.col("total_qty_sold").desc())
    .limit(5)
)

In [0]:
items_table_name = f"{CATALOG}.{SILVER_SCHEMA}.{ITEMS_TABLE}"

print(f"💾 Gravando tabela: {items_table_name}")
print("=" * 80)

# Escrever tabela Delta
items_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(items_table_name)

print(f"\n✅ Tabela {items_table_name} criada com sucesso!")
print(f"   Total de registros: {total_items:,}")

# Otimizar tabela
print(f"\n🛠️ Otimizando tabela...")
spark.sql(f"OPTIMIZE {items_table_name}")
print(f"✅ OPTIMIZE concluído!")

print(f"\n📊 Coletando estatísticas...")
spark.sql(f"ANALYZE TABLE {items_table_name} COMPUTE STATISTICS")
print(f"✅ ANALYZE TABLE concluído!")

print(f"\n🔍 Verificação da tabela criada:")
verification_df = spark.table(items_table_name)
verification_count = verification_df.count()
print(f"   Registros gravados: {verification_count:,}")

if verification_count == total_items:
    print(f"   ✅ Verificação bem-sucedida!")
else:
    print(f"   ⚠️ Divergência! Esperado: {total_items:,}, Encontrado: {verification_count:,}")

## 🖱️ 2️⃣ Explodir CLICKED_ITEMS → Tabela `sales_order_clicks`

### 🎯 Objetivo
Criar tabela com **uma linha por click** (produto clicado antes de comprar).

### 📋 Transformações
1. **Explodir array duplo**: `CLICKED_ITEMS` é `ARRAY<ARRAY<STRING>>`
2. **Extração de dados**: 
   - `[0]` = CLICKED_PRODUCT_ID
   - `[1]` = CLICK_SCORE (engajamento do usuário)
3. **Colunas calculadas**:
   - `CLICK_POSITION` = Ordem do click (1º, 2º, 3º...)

### 🔍 Estrutura dos Dados

**ANTES (array de arrays):**
```python
CLICKED_ITEMS: ARRAY<ARRAY<STRING>>
[
    ["AVpfPEx61cnluZ0-gyT9", "34"],  ← [product_id, click_score]
    ["AVpfuJ4pilAPnD_xhDyM", "98"],
    ["AVpe6jFBilAPnD_xQxO2", "60"]
]
```

**DEPOIS (tabela normalizada):**
```
COLUNAS: ORDER_NUMBER, CUSTOMER_ID, CLICKED_PRODUCT_ID, 
         CLICK_SCORE, CLICK_POSITION
```

### 💡 Análises Possíveis
- Taxa de conversão: clicks → compras
- Produtos mais clicados vs mais vendidos
- Engajamento por produto (CLICK_SCORE)
- Padrões de navegação (CLICK_POSITION)

In [0]:
print("🖱️ 2️⃣ Explodindo CLICKED_ITEMS → sales_order_clicks...")
print("=" * 80)

# Ler tabela Silver original
orders_df = spark.table(source_table_name)

print(f"\n📖 Processando clicks da tabela: {source_table_name}")

# Explodir CLICKED_ITEMS (ARRAY<ARRAY<STRING>>)
# Primeiro explode: transforma cada sub-array em uma linha
# Segundo: extrair product_id (índice 0) e click_score (índice 1)
clicks_df = orders_df.select(
    # --- COLUNAS DO PEDIDO ---
    "ORDER_NUMBER",
    "CUSTOMER_ID",
    "CUSTOMER_NAME",
    "ORDER_DATETIME",
    
    # --- EXPLODIR ARRAY DE CLICKS ---
    F.explode("CLICKED_ITEMS").alias("click"),
    
    # --- AUDITORIA ---
    "PROCESSED_AT",
    "SOURCE_TABLE",
    "PIPELINE_RUN_ID"
).select(
    # Colunas do pedido
    "ORDER_NUMBER",
    "CUSTOMER_ID",
    "CUSTOMER_NAME",
    "ORDER_DATETIME",
    
    # --- EXTRAIR ELEMENTOS DO ARRAY [product_id, click_score] ---
    F.col("click")[0].alias("CLICKED_PRODUCT_ID"),
    F.col("click")[1].cast("int").alias("CLICK_SCORE"),
    
    # Auditoria
    "PROCESSED_AT",
    "SOURCE_TABLE",
    "PIPELINE_RUN_ID"
)

# Adicionar CLICK_POSITION (ordem do click dentro de cada pedido)
clicks_df = clicks_df.withColumn(
    "CLICK_POSITION",
    F.row_number().over(Window.partitionBy("ORDER_NUMBER").orderBy(F.monotonically_increasing_id()))
)

total_clicks = clicks_df.count()
total_orders_with_clicks = clicks_df.select("ORDER_NUMBER").distinct().count()

print(f"\n✅ CLICKED_ITEMS explodido com sucesso!")
print(f"\n📊 Estatísticas:")
print(f"   Total de clicks (linhas): {total_clicks:,}")
print(f"   Total de pedidos com clicks: {total_orders_with_clicks:,}")
print(f"   Média de clicks por pedido: {total_clicks/total_orders_with_clicks:.2f}")

print(f"\n📋 Schema da tabela sales_order_clicks:")
clicks_df.printSchema()

print(f"\n📊 Amostra (10 primeiros clicks):")
display(clicks_df.limit(10))

print(f"\n📊 Distribuição de click_score:")
display(
    clicks_df.groupBy("CLICK_SCORE")
    .agg(F.count("*").alias("total_clicks"))
    .orderBy(F.col("CLICK_SCORE").desc())
    .limit(10)
)

print(f"\n📊 Top 5 produtos mais clicados:")
display(
    clicks_df.groupBy("CLICKED_PRODUCT_ID")
    .agg(
        F.count("*").alias("total_clicks"),
        F.avg("CLICK_SCORE").alias("avg_click_score")
    )
    .orderBy(F.col("total_clicks").desc())
    .limit(5)
)

In [0]:
clicks_table_name = f"{CATALOG}.{SILVER_SCHEMA}.{CLICKS_TABLE}"

print(f"💾 Gravando tabela: {clicks_table_name}")
print("=" * 80)

# Escrever tabela Delta
clicks_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(clicks_table_name)

print(f"\n✅ Tabela {clicks_table_name} criada com sucesso!")
print(f"   Total de registros: {total_clicks:,}")

# Otimizar tabela
print(f"\n🛠️ Otimizando tabela...")
spark.sql(f"OPTIMIZE {clicks_table_name}")
print(f"✅ OPTIMIZE concluído!")

print(f"\n📊 Coletando estatísticas...")
spark.sql(f"ANALYZE TABLE {clicks_table_name} COMPUTE STATISTICS")
print(f"✅ ANALYZE TABLE concluído!")

print(f"\n🔍 Verificação da tabela criada:")
verification_clicks = spark.table(clicks_table_name)
verification_clicks_count = verification_clicks.count()
print(f"   Registros gravados: {verification_clicks_count:,}")

if verification_clicks_count == total_clicks:
    print(f"   ✅ Verificação bem-sucedida!")
else:
    print(f"   ⚠️ Divergência! Esperado: {total_clicks:,}, Encontrado: {verification_clicks_count:,}")

## 📈 3️⃣ Análise de Conversão: Clicks → Compras

### 🎯 Objetivo
Analisar o comportamento do usuário:
- Quais produtos foram **clicados MAS NÃO comprados**?
- Qual a **taxa de conversão** de clicks para compras?
- Produtos com **alto engajamento** (click_score) mas **baixa conversão**?

### 📊 Métricas Importantes
1. **Taxa de conversão global**: % de clicks que viraram compras
2. **Taxa de conversão por produto**: Quais produtos convertem melhor
3. **Produtos abandonados**: Clicks sem compra (oportunidade de marketing!)
4. **Engajamento médio**: CLICK_SCORE dos produtos mais clicados

In [0]:
print("📈 ANÁLISE DE CONVERSÃO: CLICKS → COMPRAS")
print("=" * 80)

# Ler tabelas criadas
items_tbl = spark.table(items_table_name)
clicks_tbl = spark.table(clicks_table_name)

print(f"\n📊 1. Taxa de Conversão Global")
print("   (% de clicks que resultaram em compra)\n")

# LEFT JOIN: clicks com compras
conversion_df = clicks_tbl.alias("c").join(
    items_tbl.alias("i"),
    (F.col("c.ORDER_NUMBER") == F.col("i.ORDER_NUMBER")) & 
    (F.col("c.CLICKED_PRODUCT_ID") == F.col("i.PRODUCT_ID")),
    "left"
)

total_clicks_unique = clicks_tbl.count()
total_converted = conversion_df.filter(F.col("i.PRODUCT_ID").isNotNull()).count()
conversion_rate = (total_converted / total_clicks_unique) * 100

print(f"   Total de clicks: {total_clicks_unique:,}")
print(f"   Clicks que viraram compra: {total_converted:,}")
print(f"   Taxa de conversão: {conversion_rate:.2f}%")

print(f"\n📊 2. Top 10 Produtos: Mais Clicados vs Mais Comprados")

# Produtos mais clicados
top_clicked = clicks_tbl.groupBy("CLICKED_PRODUCT_ID") \
    .agg(
        F.count("*").alias("total_clicks"),
        F.avg("CLICK_SCORE").alias("avg_engagement")
    ) \
    .orderBy(F.col("total_clicks").desc()) \
    .limit(10)

print("\n   🖱️ Produtos Mais Clicados:")
display(top_clicked)

# Produtos mais comprados
top_purchased = items_tbl.groupBy("PRODUCT_ID", "PRODUCT_NAME") \
    .agg(
        F.sum("QUANTITY").alias("total_qty_sold"),
        F.count("*").alias("times_ordered")
    ) \
    .orderBy(F.col("total_qty_sold").desc()) \
    .limit(10)

print("\n   🛒 Produtos Mais Comprados:")
display(top_purchased)

print(f"\n📊 3. Produtos Abandonados (Clicados mas NÃO Comprados)")

# LEFT ANTI JOIN: clicks que NÃO viraram compra
abandoned_df = clicks_tbl.alias("c").join(
    items_tbl.alias("i"),
    (F.col("c.ORDER_NUMBER") == F.col("i.ORDER_NUMBER")) & 
    (F.col("c.CLICKED_PRODUCT_ID") == F.col("i.PRODUCT_ID")),
    "left_anti"
)

abandoned_products = abandoned_df.groupBy("CLICKED_PRODUCT_ID") \
    .agg(
        F.count("*").alias("abandoned_clicks"),
        F.avg("CLICK_SCORE").alias("avg_engagement")
    ) \
    .orderBy(F.col("abandoned_clicks").desc()) \
    .limit(10)

print(f"\n   🚫 Top 10 Produtos Abandonados (Alto engajamento, sem compra):")
print(f"   💡 Oportunidade de remarketing!\n")
display(abandoned_products)

print(f"\n📊 4. Taxa de Conversão por Produto")

# Para cada produto: clicks vs compras
product_conversion = clicks_tbl.groupBy("CLICKED_PRODUCT_ID").agg(
    F.count("*").alias("total_clicks")
).alias("clicks").join(
    items_tbl.groupBy("PRODUCT_ID").agg(
        F.count("*").alias("total_purchases")
    ).alias("items"),
    F.col("clicks.CLICKED_PRODUCT_ID") == F.col("items.PRODUCT_ID"),
    "left"
).select(
    F.col("clicks.CLICKED_PRODUCT_ID").alias("PRODUCT_ID"),
    F.col("total_clicks"),
    F.coalesce(F.col("total_purchases"), F.lit(0)).alias("total_purchases"),
    ((F.coalesce(F.col("total_purchases"), F.lit(0)) / F.col("total_clicks")) * 100).alias("conversion_rate")
).orderBy(F.col("total_clicks").desc())

print(f"\n   Top 10 Produtos por Taxa de Conversão (com > 5 clicks):\n")
display(
    product_conversion.filter(F.col("total_clicks") >= 5)
    .orderBy(F.col("conversion_rate").desc())
    .limit(10)
)

## 📊 Resumo Final - Tabelas Criadas

### ✅ Tabelas Silver Normalizadas

| Tabela | Tipo | Granularidade | Registros | Colunas | Uso Principal |
|--------|------|---------------|-----------|---------|---------------|
| **sales_orders** | FATO AGREGADO | 1 linha = 1 pedido | 4.000 | 12 | Análises agregadas, dashboards de vendas |
| **sales_order_items** ⭐ | FATO DETALHADO | 1 linha = 1 item | ~8.000 | 22 | Análises por produto, receita por SKU |
| **sales_order_clicks** | EVENTO | 1 linha = 1 click | ~16.000 | 10 | Análise de comportamento, conversão |

---

### 🔗 Relacionamentos (Star Schema)

```
           ┌─────────────┐
           │  customers  │
           └──────┬──────┘
                  │
              CUSTOMER_ID
                  │
                  ▼
          ┌───────────────┐
          │ sales_orders  │
          │  (FATO PAI)   │
          └───────┬───────┘
                  │
           ORDER_NUMBER (FK)
                  │
         ┌────────┴────────┐
         │                 │
         ▼                 ▼
┌─────────────────┐  ┌──────────────┐
│ sales_order_    │  │ sales_order_ │
│ items           │  │ clicks       │
│ (FATO FILHO)    │  │ (EVENTO)     │
└────────┬────────┘  └──────────────┘
         │
    PRODUCT_ID (FK)
         │
         ▼
   ┌──────────┐
   │ products │
   └──────────┘
```

---

### 📈 Queries Típicas

#### 1. Vendas por Produto (SKU-level)
```sql
SELECT 
    PRODUCT_ID,
    PRODUCT_NAME,
    SUM(ITEM_TOTAL) as total_revenue,
    SUM(QUANTITY) as total_qty_sold,
    COUNT(DISTINCT ORDER_NUMBER) as times_ordered
FROM retail_dev.silver.sales_order_items
GROUP BY PRODUCT_ID, PRODUCT_NAME
ORDER BY total_revenue DESC;
```

#### 2. Taxa de Conversão: Clicks → Compras
```sql
SELECT 
    c.CLICKED_PRODUCT_ID,
    COUNT(DISTINCT c.ORDER_NUMBER) as total_clicks,
    COUNT(DISTINCT i.ORDER_NUMBER) as total_purchases,
    (COUNT(DISTINCT i.ORDER_NUMBER) / COUNT(DISTINCT c.ORDER_NUMBER)) * 100 as conversion_rate
FROM retail_dev.silver.sales_order_clicks c
LEFT JOIN retail_dev.silver.sales_order_items i
    ON c.ORDER_NUMBER = i.ORDER_NUMBER 
    AND c.CLICKED_PRODUCT_ID = i.PRODUCT_ID
GROUP BY c.CLICKED_PRODUCT_ID
ORDER BY total_clicks DESC;
```

#### 3. Análise Completa de Pedido
```sql
SELECT 
    o.ORDER_NUMBER,
    o.CUSTOMER_NAME,
    o.ORDER_DATETIME,
    i.PRODUCT_NAME,
    i.QUANTITY,
    i.ITEM_TOTAL,
    i.HAS_ITEM_PROMO
FROM retail_dev.silver.sales_orders o
JOIN retail_dev.silver.sales_order_items i
    ON o.ORDER_NUMBER = i.ORDER_NUMBER
WHERE o.CUSTOMER_ID = 12345;
```

---

### 💡 Próximos Passos Sugeridos

1. **Camada Gold**: Criar métricas agregadas (vendas diárias, por categoria, por cliente)
2. **Enriquecer com Dimensões**: JOIN com `products`, `customers`, `loyalty_segments`
3. **Dashboard BI**: Visualizações de conversão, top produtos, receita
4. **Alertas**: Produtos com baixa conversão, estoque crítico
5. **ML**: Recomendação de produtos baseada em clicks

In [0]:
print("🔍 VERIFICAÇÃO FINAL - TODAS AS TABELAS SILVER")
print("=" * 80)

# Lista de tabelas Silver relacionadas a sales_orders
tables = [
    f"{CATALOG}.{SILVER_SCHEMA}.{SOURCE_TABLE}",
    f"{CATALOG}.{SILVER_SCHEMA}.{ITEMS_TABLE}",
    f"{CATALOG}.{SILVER_SCHEMA}.{CLICKS_TABLE}"
]

print(f"\n📋 Resumo das Tabelas Silver:")
print(f"\n{'Tabela':<40} {'Registros':>15} {'Colunas':>10}")
print("=" * 80)

for table in tables:
    try:
        df = spark.table(table)
        count = df.count()
        cols = len(df.columns)
        table_short = table.split(".")[-1]  # Nome curto
        print(f"{table_short:<40} {count:>15,} {cols:>10}")
    except Exception as e:
        table_short = table.split(".")[-1]
        print(f"{table_short:<40} {'ERRO':>15} {'-':>10}")
        print(f"   ⚠️ Erro: {e}")

print("\n" + "="*80)
print(f"✅ EXPANSÃO DE ARRAYS CONCLUÍDA COM SUCESSO! 🎉")
print("="*80)

print(f"\n💡 Dicas de Uso:")
print(f"   • sales_orders: Análises agregadas de pedidos (KPIs gerais)")
print(f"   • sales_order_items: Análises granulares por produto/SKU")
print(f"   • sales_order_clicks: Análises de comportamento e conversão")
print(f"\n🔗 Todas as tabelas têm FK ORDER_NUMBER para JOINs")
print(f"📊 Change Data Feed habilitado para auditoria de mudanças")
print(f"⚡ Tabelas otimizadas (OPTIMIZE + ANALYZE TABLE)")

print(f"\n📈 Próximos passos sugeridos:")
print(f"   1. Criar camada Gold com métricas agregadas")
print(f"   2. JOIN com dimensões (products, customers, loyalty_segments)")
print(f"   3. Criar dashboard BI de conversão e vendas")
print(f"   4. Implementar alertas de produtos com baixa conversão")